# Phase 6 - Abstractive Summarization: all experiments in one notebook

**Run All** runs every experiment one after another. Everything is cached: a finished training is loaded, finished
summaries / scores / zips are skipped, an interrupted training resumes from its last checkpoint (every 250 optimiser
steps) and an interrupted generation continues where it stopped (saved every 10 documents). You can stop and
re-run this notebook at any time.

Pipeline (`src/summ_utils.py`, `src/summ_models.py`, `src/summ_experiment.py`):

    document -> Phase-1 cleaning (cased) -> bpe32k tokens
      decoder-only (A2, A, B): [first 900 document tokens] [EOS] [summary] [EOS]   (B reads document + EOS bidirectionally)
      encoder-decoder (D)    : encoder <- first 1024 tokens (or 3 chunks of 1024), decoder <- [EOS] summary
    loss   = negative log-likelihood of the summary tokens only (A2 / D: with the copy mechanism)
    target = first 620 summary tokens (BERTScore only reads ~426 words of the reference, P6_01)
    best epoch = lowest validation loss (367 held-out documents)
    generation: greedy, KV cache, copy mechanism, no repeated 5-gram inside the summary, EOS allowed after half the
                target length, target length = 4.62 * doc_words^0.612 words (P6_01 rule), capped at 500 words
    local evaluation: 150 held-out documents, BERTScore F1 (roberta-large, as the course evaluator) + ROUGE-1 / ROUGE-L
    test: 100 summaries -> submission.zip in the run folder

Decisions: PROJECT_NOTES section 13.3.


In [1]:
import os, sys, json, time, traceback
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import torch
import summ_utils as su
from summ_experiment import Experiment, compare
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)
print("project root:", ROOT, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

# QUICK_TEST: tiny run of the whole notebook (used for the CPU test); keep False for the real run
QUICK_TEST = os.environ.get("P6_QUICK_TEST") == "1"
QUICK = dict(train_limit=48, val_limit=8, eval_limit=4, test_limit=3, epochs=1, ckpt_every=2, max_words=40) if QUICK_TEST else {}
print("QUICK_TEST:", QUICK_TEST)

project root: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project | torch 2.14.0+cu132 | cuda True
QUICK_TEST: False


c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. The experiments

| #   | Phase-1 model                         | technique (doc 12.2.1 list)                                             | input                                 | why                                                                                                                                            |
| --- | ------------------------------------- | ----------------------------------------------------------------------- | ------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------- |
| 1   | A2 decoder + copy                     | **full fine-tuning** (not in the list; KD teacher + reference for PEFT) | first 900 tokens                      | best Phase-1 model; copy fits summaries that re-use 94% of the document's tokens                                                               |
| 2   | B prefix-LM                           | **LoRMA** r8 (compulsory)                                               | first 900, read bidirectionally       | multiplicative low-rank update; B was pretrained to read a prefix in both directions                                                           |
| 3   | A plain decoder                       | **Houlsby adapters** (64)                                               | first 900                             | new small layers after attention and feed-forward                                                                                              |
| 4   | D encoder-decoder + copy              | **LoRA** r8                                                             | encoder: first 1024                   | encoder-decoder: the summary does not take document space                                                                                      |
| 5   | D encoder-decoder + copy              | LoRA r8 + **fusion-in-decoder** (long document)                         | encoder: 3 chunks x 1024 = first 3072 | P6_01: first 3072 tokens cover 64% of the visible summary bigrams vs 51% for 1536; same as #4 except the input -> clean long-document ablation |
| 6   | D-small (decoder layers 0, 2, 4 of 5) | **knowledge distillation** from #1 (word-level, alpha 0.5)              | encoder: first 1024                   | smaller + faster student (60M vs 70M; 3 vs 10 layers run per generated token) learns the teacher's token distributions                         |
| 6c  | D-small                               | same, **without** teacher (control, not submitted)                      | first 1024                            | shows what KD adds                                                                                                                             |

Extra analysis on #1 (no extra training): BERTScore / ROUGE when its summaries are cut to 128 / 256 / 426 words
(doc 12.5: "how generation length affects BERTScore and ROUGE").

Rough time on the RTX 4050: training ~10-15 min per epoch (KD ~+40%, D faster), generation ~5-8 s per summary
(150 local + 100 test per experiment) -> about 7-9 hours for everything.


In [2]:
TEACHER = {"phase1": "A2", "method": "full"}
EXPERIMENTS = [
    # (label, config, make a test submission?)
    ("#1 A2 full fine-tuning (KD teacher)", TEACHER, True),
    ("#2 B + LoRMA r8", {"phase1": "B", "method": "lorma"}, True),
    ("#3 A + Houlsby adapters 64", {"phase1": "A", "method": "houlsby"}, True),
    ("#4 D + LoRA r8, encoder 1 x 1024", {"phase1": "D", "method": "lora"}, True),
    ("#5 D + LoRA r8, encoder 3 x 1024 (fusion-in-decoder)", {"phase1": "D", "method": "lora", "n_chunks": 3}, True),
    ("#6 D-small + KD from #1", {"phase1": "D", "method": "full", "keep_dec_layers": [0, 2, 4], "epochs": 4, "kd_teacher": TEACHER}, True),
    ("#6c D-small without KD (control)", {"phase1": "D", "method": "full", "keep_dec_layers": [0, 2, 4], "epochs": 4}, False),
]
if QUICK_TEST:
    TEACHER.update(QUICK)
    EXPERIMENTS = [(l, {**c, **QUICK}, s) for l, c, s in EXPERIMENTS]

def run(label, cfg, submit):
    print(f"\n{'=' * 110}\n{time.strftime('%Y-%m-%d %H:%M')}  {label}  {cfg}\n{'=' * 110}")
    exp = None
    try:
        exp = Experiment(ROOT, cfg)
        exp.memory_check()
        exp.train()
        exp.evaluate_local(lengths_analysis=label.startswith("#1"))
        if submit:
            exp.predict_test()
        exp.log()
        print(f"{time.strftime('%H:%M')}  {label} finished")
    except Exception:
        print(f"{time.strftime('%H:%M')}  {label} FAILED:")
        traceback.print_exc(file=sys.stdout)
    finally:
        del exp
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 2. Run everything

Each experiment prints: memory check, parameters (trainable share), validation loss before fine-tuning and after every
epoch, local-eval scores, submission check. A failure in one experiment is printed and the next one starts.


In [3]:
for label, cfg, submit in EXPERIMENTS:
    run(label, cfg, submit)
print("\nall experiments done", time.strftime("%Y-%m-%d %H:%M"))


2026-10-29 18:38  #1 A2 full fine-tuning (KD teacher)  {'phase1': 'A2', 'method': 'full'}


tokenising test: 100%|██████████| 1/1 [00:00<00:00,  2.33it/s]


data: train 6661, val 367, local_eval 150, test 100 | targets cut at 620 tokens: 62%
experiment: A2_full_doc900 | device cuda | precision torch.bfloat16
memory check: {'params_M': 70.22, 'trainable_M': 70.22, 'trainable_pct': 100.0, 'micro_batch': '2 x 1522', 'sec': 0.9, 'peak_gb': 2.22}
parameters: 70.22M, trainable 70.220M (100.00%)


before fine-tuning: val loss 2.3927


epoch 1/3: 100%|██████████| 417/417 [07:02<00:00,  1.01s/it, nll=2.145, copy=0.51]


epoch 1 | train NLL 2.1717 | val loss 2.1490 | 7 min  <- best


... (80 outputs left out)

epoch 4/4: 100%|██████████| 417/417 [05:16<00:00,  1.32it/s, nll=2.059, copy=0.45]


epoch 4 | train NLL 2.0395 | val loss 2.4200 | 6 min
early stop: no improvement for 1 epoch(s)
training finished -> best epoch 3 (val loss 2.4148)


Loading weights: 100%|██████████| 389/389 [00:01<00:00, 302.47it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


local eval: {'n': 150, 'rouge1_f1': 0.3744, 'rouge2_f1': 0.1557, 'rougeL_f1': 0.2102, 'pred_words': 448.46, 'bertscore_p': 0.8389, 'bertscore_r': 0.8283, 'bertscore_f1': 0.8335, 'repeated_4gram_pct': 27.1664, 'novel_2gram_pct': 32.0852, 'ref_novel_2gram_pct': 28.7816, 'copy_weight': 0.564, 'stop_reasons': {'eos': 25, 'max_words': 125}, 'sec_per_doc': 2.6506, 'gen_tokens_per_s': 199.7359, 'gen_peak_gb': 0.32}
logged: {'run': 'Dsmall_dec024_full_enc1x1024', 'phase1': 'D', 'method': 'full', 'kd': False, 'params_M': 60.26, 'trainable_M': 60.265, 'trainable_pct': 100.0, 'train_peak_gb': 0.96, 'best_epoch': 3, 'epochs_run': 4, 'min_per_epoch': 5.6, 'val_loss_before': 3.5996897091198394, 'val_loss': 2.414822439439191, 'bertscore_f1': 0.8335101008415222, 'bertscore_p': 0.8389310240745544, 'bertscore_r': 0.8282701373100281, 'rouge1_f1': 0.3743899249631403, 'rouge2_f1': 0.15567400066970552, 'rougeL_f1': 0.2101729892826613, 'pred_words': 448.46, 'repeated_4gram_pct': 27.166355330398012, 'novel_2g

## 3. Comparison (local-eval set, 150 held-out documents)

Reference points from P6_01 (same 150 documents): same summary for every case 0.804 BERTScore, lead-512 words 0.832,
**random sentences 0.842**, oracle extractive (uses the reference) 0.873.


In [4]:
tbl = compare(ROOT)
display(tbl)
base = json.load(open(ROOT / "experiments" / "phase6" / "p6_01_data" / "baselines.json"))
display(pd.DataFrame({k: base[k] for k in ["same summary for every case", "lead-512", "random sentences, predicted length",
                                           "ORACLE extractive (uses reference)"] if k in base}).T[
    ["pred_words", "bertscore_f1", "rouge1_f1", "rougeL_f1"]].round(4).rename_axis("P6_01 baselines"))

,phase1,method,kd,params_M,trainable_pct,val_loss_before,val_loss,bertscore_f1,rouge1_f1,rougeL_f1,pred_words,repeated_4gram_pct,novel_2gram_pct,copy_weight,min_per_epoch,sec_per_doc,train_peak_gb
run,,,,,,,,,,,,,,,,,
A2_full_doc900,A2,full,False,70.22,100.00,2.392684,2.136509,0.834689,0.349087,0.201810,351.700000,22.180836,30.416112,0.690664,7.2,4.507267,2.22
B_lormar8_doc900,B,lorma,False,71.12,1.50,2.879566,2.497840,0.819105,0.322468,0.194883,418.600000,31.850747,49.015856,NaN,7.2,7.851800,2.16
A_houlsby64_doc900,A,houlsby,False,71.71,2.30,2.738681,2.384425,0.823049,0.323520,0.194836,409.760000,30.543145,47.915830,NaN,5.1,6.465333,1.72
D_lorar8_enc1x1024,D,lora,False,74.43,1.52,2.728139,2.382656,0.828904,0.350603,0.205049,412.473333,28.973197,35.382211,0.521414,6.8,5.788067,1.07
D_lorar8_enc3x1024,D,lora,False,74.43,1.52,2.537314,1.788400,0.839663,0.399323,0.219772,420.460000,22.345636,22.416732,0.678534,10.0,5.876800,1.37
Dsmall_dec024_full_enc1x1024_kd0.5,D,full,True,60.26,100.00,3.599690,2.369722,0.831775,0.346701,0.200211,384.306667,27.195119,34.442231,0.522063,9.5,2.240333,1.13
Dsmall_dec024_full_enc1x1024,D,full,False,60.26,100.00,3.599690,2.414822,0.833510,0.374390,0.210173,448.460000,27.166355,32.085245,0.563998,5.6,2.650600,0.96


,pred_words,bertscore_f1,rouge1_f1,rougeL_f1
P6_01 baselines,,,,
same summary for every case,638.0000,0.8043,0.3173,0.1576
lead-512,509.8467,0.8317,0.4147,0.2084
"random sentences, predicted length",799.6800,0.8418,0.4995,0.2486
ORACLE extractive (uses reference),1090.9000,0.8734,0.6836,0.4226


## 4. Knowledge distillation: quality vs size and speed (#1 teacher, #6 student, #6c control)


In [5]:
if tbl is not None:
    kd_rows = [r for r in tbl.index if r.startswith("A2_full") or r.startswith("Dsmall")]
    cols = [c for c in ["params_M", "trainable_pct", "val_loss", "bertscore_f1", "rouge1_f1", "rougeL_f1", "sec_per_doc",
                        "min_per_epoch", "train_peak_gb"] if c in tbl]
    display(tbl.loc[kd_rows, cols])

,params_M,trainable_pct,val_loss,bertscore_f1,rouge1_f1,rougeL_f1,sec_per_doc,min_per_epoch,train_peak_gb
run,,,,,,,,,
A2_full_doc900,70.22,100.0,2.136509,0.834689,0.349087,0.201810,4.507267,7.2,2.22
Dsmall_dec024_full_enc1x1024_kd0.5,60.26,100.0,2.369722,0.831775,0.346701,0.200211,2.240333,9.5,1.13
Dsmall_dec024_full_enc1x1024,60.26,100.0,2.414822,0.833510,0.374390,0.210173,2.650600,5.6,0.96


## 5. Generation length (summaries of #1 cut to 128 / 256 / 426 words)


In [6]:
t_exp = Experiment(ROOT, TEACHER, quiet=True)
p = t_exp.dir / "local_scores.json"
if p.exists() and "by_length" in json.load(open(p)):
    bl = json.load(open(p))["by_length"]
    display(pd.DataFrame(bl).T[["pred_words", "bertscore_f1", "rouge1_f1", "rouge2_f1", "rougeL_f1"]].round(4).rename_axis("cut to words"))

,pred_words,bertscore_f1,rouge1_f1,rouge2_f1,rougeL_f1
cut to words,,,,,
128,127.8400,0.8358,0.2220,0.1006,0.1438
256,252.3667,0.8365,0.3162,0.1368,0.1884
426,336.9067,0.8349,0.3466,0.1465,0.2009
full,351.7000,0.8347,0.3491,0.1467,0.2018


## 6. Examples: reference vs every model (first 2 local-eval documents, first 90 words)


In [7]:
split = json.load(open(ROOT / "data" / "processed" / "phase6" / "split_v1.json"))
train_docs = {d["id"]: d for d in su.load_summ(ROOT)[0]["train"]}
for doc_id in split["local_eval"][:2]:
    print(f"\n######## document {doc_id} ({len(train_docs[doc_id]['document'].split())} words)")
    print("REFERENCE:", " ".join(su.clean_summ(train_docs[doc_id]["summary"]).split()[:90]), "...")
    for label, cfg, _ in EXPERIMENTS:
        e = Experiment(ROOT, cfg, quiet=True)
        f = e.dir / "local_preds.json"
        if f.exists():
            pr = json.load(open(f)).get(doc_id)
            if pr:
                print(f"\n{label}: ({len(pr['summary'].split())} words, stop: {pr['stop']})")
                print("  ", " ".join(pr["summary"].split()[:90]), "...")


######## document 2434 (1921 words)
REFERENCE: The appellant claimed that certain property comprised in an Inam which was abolished by virtue of the Andhra Inams (Abolition & Conversion into Ryotwari) Act 36 of 1956 should be registered in his name. His contention was that prior to its abolition he was the Inamdar of that Inam though he had the liability to repair a tank in his village from out of the income of Inam was granted for a charitable purpose, the object of assuming the Inam was granted for a charitable purpose, the. object of the ...

#1 A2 full fine-tuning (KD teacher): (319 words, stop: eos)
   The appellant was the Manager of a tank which was a charitable institution. The Inam was granted to the appellant. The appellant was shown to be the Manager of the tank. The Inam was granted in favour of the tank and the appellant was in possession of the Inam property. The appellant was in possession of the Inam. The Inam was granted in favour by the Collector. The appellant was i